# White-Box Lie Detection - Llama-3.1-70B-Instruct (Colab Pro / A100)

Ce notebook lance l'experience de lie detection white-box sur **Llama-3.1-70B-Instruct** en utilisant un GPU A100 (Colab Pro).

**Projet base sur** [llm_lie_detection_black_vs_white_box](https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box) par GeorgeVasile04.

## Metrique principale : **AUC** (ROC AUC)
## Selection du best layer : **4-fold Stratified Cross-Validation** (pas de data leakage)

## Comment utiliser
1. Ouvre ce notebook dans Google Colab
2. Va dans **Runtime > Change runtime type > GPU (A100)** (Colab Pro requis)
3. Accepte la licence Llama sur HuggingFace : https://huggingface.co/meta-llama/Llama-3.1-70B-Instruct
4. Cree un token sur https://huggingface.co/settings/tokens
5. Execute toutes les cellules dans l'ordre

| Modele | Params | VRAM (4-bit) | GPU requis |
|--------|--------|-------------|------------|
| Llama-3.1-70B-Instruct | 70B | ~38GB | A100 (40GB) |

## 1. Setup: GPU check + installations

In [ ]:
# Verifier le GPU disponible
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_vram = props.total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_vram:.1f} GB")
    if gpu_vram < 35:
        print("=" * 50)
        print("  ATTENTION: Llama-2-70b-chat en 4-bit a besoin d'environ 38GB VRAM.")
        print("  Tu as seulement {:.0f}GB. Il faut un A100 (40GB).".format(gpu_vram))
        print("  Va dans Runtime > Change runtime type > A100")
        print("=" * 50)
else:
    print("PAS DE GPU! Va dans Runtime > Change runtime type > GPU (A100)")
    raise RuntimeError("GPU requis")

In [ ]:
# Installer les dependances
!pip install -U -q "transformers>=4.49" accelerate bitsandbytes datasets scikit-learn matplotlib pandas numpy tqdm huggingface_hub

# Redemarrer le runtime apres l'install pour que numpy se recharge correctement
import os
if not os.environ.get("DEPS_INSTALLED"):
    os.environ["DEPS_INSTALLED"] = "1"
    print("=" * 50)
    print("  Dependances installees!")
    print("  Le runtime va redemarrer automatiquement.")
    print("  -> Relance TOUTES les cellules apres le restart")
    print("  (Runtime > Run all, ou Ctrl+F9)")
    print("=" * 50)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("Runtime deja redemarre, on continue...")

In [ ]:
# Monter Google Drive pour sauvegarder les resultats
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/LLM_Lie_Detection_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Resultats sauvegardes dans: {RESULTS_DIR}")

In [ ]:
# Login HuggingFace - OBLIGATOIRE pour Llama (licence Meta)
# 1. Accepte la licence sur https://huggingface.co/meta-llama/Llama-3.1-70B-Instruct
# 2. Cree un token sur https://huggingface.co/settings/tokens (type "Read")
# 3. Colle ton token ci-dessous quand demande
from huggingface_hub import login
login()

## 2. Code principal

In [ ]:
import gc
import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

print("Imports OK")

In [ ]:
#######################################################################
# CONFIGURATION
#######################################################################

MODEL_CONFIG = {
    "name": "Llama-3.1-70B-Instruct",
    "hf_id": "meta-llama/Llama-3.1-70B-Instruct",
    "num_params": 70.0,
    "quant": "4bit",
}

# Memes datasets que les experiences locales (Pythia)
DATASETS_CONFIG = [
    {"name": "arc_easy",        "hf_id": "allenai/ai2_arc", "subset": "ARC-Easy",  "type": "arc"},
    {"name": "common_sense_qa", "hf_id": "tau/commonsense_qa", "subset": None,      "type": "csqa"},
    {"name": "boolq",           "hf_id": "google/boolq",    "subset": None,         "type": "boolq"},
    {"name": "imdb",            "hf_id": "stanfordnlp/imdb", "subset": None,         "type": "imdb"},
    {"name": "dbpedia_14",      "hf_id": "fancyzhx/dbpedia_14", "subset": None,     "type": "dbpedia"},
]

MAX_TRAIN = 100
MAX_VAL = 200
LAYERS_SKIP = 2   # Extraire 1 couche sur 2
N_FOLDS = 4       # 4-fold CV pour best layer (comme George)

print(f"Modele: {MODEL_CONFIG['name']} ({MODEL_CONFIG['num_params']}B)")
print(f"Datasets: {[d['name'] for d in DATASETS_CONFIG]}")
print(f"4-fold CV pour selection du best layer")
print(f"Metrique principale: AUC (ROC AUC)")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU VRAM: {vram_gb:.1f} GB")

In [ ]:
#######################################################################
# Dataset Loading - paires Vrai/Faux
#######################################################################

def load_binary_dataset(config):
    """Load a dataset and convert to True/False text statements."""
    ds_type = config["type"]
    samples = {"train": [], "validation": []}

    if ds_type == "arc":
        ds = load_dataset(config["hf_id"], config["subset"])
        for split_name, split_key in [("train", "train"), ("validation", "test")]:
            for row in ds[split_key]:
                q = row["question"]
                choices = row["choices"]["text"]
                labels = row["choices"]["label"]
                correct_idx = labels.index(row["answerKey"]) if row["answerKey"] in labels else 0
                correct = choices[correct_idx]
                wrong = choices[(correct_idx + 1) % len(choices)]
                group_id = q[:50]
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {correct}\nIs this answer correct? Yes.", "label": True, "group_id": group_id})
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {wrong}\nIs this answer correct? No.", "label": False, "group_id": group_id})

    elif ds_type == "csqa":
        ds = load_dataset(config["hf_id"])
        for split_name, split_key in [("train", "train"), ("validation", "validation")]:
            for row in ds[split_key]:
                q = row["question"]
                choices = row["choices"]["text"]
                labels = row["choices"]["label"]
                answer_key = row.get("answerKey", "A")
                correct_idx = labels.index(answer_key) if answer_key in labels else 0
                correct = choices[correct_idx]
                wrong = choices[(correct_idx + 1) % len(choices)]
                group_id = q[:50]
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {correct}\nIs this answer correct? Yes.", "label": True, "group_id": group_id})
                samples[split_name].append({"text": f"Question: {q}\nAnswer: {wrong}\nIs this answer correct? No.", "label": False, "group_id": group_id})

    elif ds_type == "boolq":
        ds = load_dataset(config["hf_id"])
        for split_name in ["train", "validation"]:
            for row in ds[split_name]:
                answer = "Yes" if row["answer"] else "No"
                samples[split_name].append({"text": f"Passage: {row['passage'][:300]}\nQuestion: {row['question']}\nAnswer: {answer}", "label": row["answer"], "group_id": None})

    elif ds_type == "imdb":
        ds = load_dataset(config["hf_id"])
        for split_name, split_key in [("train", "train"), ("validation", "test")]:
            for row in ds[split_key]:
                sentiment = "positive" if row["label"] == 1 else "negative"
                samples[split_name].append({"text": f"Review: {row['text'][:400]}\nSentiment: {sentiment}", "label": row["label"] == 1, "group_id": None})

    elif ds_type == "dbpedia":
        ds = load_dataset(config["hf_id"])
        label_names = ds["train"].features["label"].names
        for split_name, split_key in [("train", "train"), ("validation", "test")]:
            for row in ds[split_key]:
                true_label = label_names[row["label"]]
                wrong_label = label_names[(row["label"] + 1) % len(label_names)]
                text_snippet = row["content"][:300]
                samples[split_name].append({"text": f"Text: {text_snippet}\nCategory: {true_label}", "label": True, "group_id": None})
                samples[split_name].append({"text": f"Text: {text_snippet}\nCategory: {wrong_label}", "label": False, "group_id": None})

    # Limit and shuffle
    for split in ["train", "validation"]:
        np.random.seed(42)
        np.random.shuffle(samples[split])
        limit = MAX_TRAIN if split == "train" else MAX_VAL
        samples[split] = samples[split][:limit]

    return samples

In [ ]:
#######################################################################
# Model Loading with 4-bit quantization
#######################################################################

def load_model(model_config):
    """Load a model with 4-bit quantization for GPU inference."""
    hf_id = model_config["hf_id"]
    print(f"Loading {hf_id}...")

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Trouver les couches transformer automatiquement
    layers = None
    for attr_path in ["model.layers", "transformer.h", "gpt_neox.layers"]:
        obj = model
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            layers = obj
            print(f"  Found {len(layers)} layers via {attr_path}")
            break
        except AttributeError:
            continue

    if layers is None:
        raise ValueError(f"Cannot find transformer layers in {hf_id}")

    model.eval()
    vram_used = torch.cuda.memory_allocated() / 1024**3
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  VRAM used: {vram_used:.1f} GB / {vram_total:.1f} GB")

    return model, tokenizer, layers

In [ ]:
#######################################################################
# Activation Extraction
#######################################################################

@torch.inference_mode()
def extract_activations(model, tokenizer, layers, text, layers_skip=2):
    """Extract hidden-state activations at each layer for the last token."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    layer_indices = list(range(0, len(layers), layers_skip))
    activations = {}
    hooks = []

    for idx in layer_indices:
        name = f"h{idx}"
        def make_hook(layer_name):
            def hook_fn(module, input, output):
                if isinstance(output, tuple):
                    hidden = output[0]
                else:
                    hidden = output
                activations[layer_name] = hidden[0, -1, :].detach().float().cpu().numpy()
            return hook_fn

        h = layers[idx].register_forward_hook(make_hook(name))
        hooks.append(h)

    model(**inputs)

    for h in hooks:
        h.remove()

    return activations

In [ ]:
#######################################################################
# Probe Algorithms
#######################################################################

@dataclass
class ProbeResult:
    direction: np.ndarray

    def predict(self, activations):
        return activations @ self.direction


def train_dim(X, y):
    """Difference in Means."""
    true_mean = X[y].mean(axis=0)
    false_mean = X[~y].mean(axis=0)
    d = true_mean - false_mean
    d = d / (np.linalg.norm(d) + 1e-8)
    return ProbeResult(direction=d)


def train_lda(X, y):
    """Linear Discriminant Analysis."""
    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)
        d = lda.coef_[0]
        d = d / (np.linalg.norm(d) + 1e-8)
        return ProbeResult(direction=d)
    except Exception:
        return train_dim(X, y)


def train_lr(X, y):
    """Logistic Regression."""
    lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
    lr.fit(X, y)
    d = lr.coef_[0]
    d = d / (np.linalg.norm(d) + 1e-8)
    return ProbeResult(direction=d)


def train_pca(X, y):
    """PCA - unsupervised."""
    pca = PCA(n_components=1)
    pca.fit(X)
    d = pca.components_[0]
    d = d / (np.linalg.norm(d) + 1e-8)
    return ProbeResult(direction=d)


def train_lat(X, y):
    """Linear Artificial Tomography."""
    np.random.seed(42)
    n = len(X)
    indices = np.random.permutation(n)
    half = n // 2
    diffs = X[indices[:half]] - X[indices[half:2*half]]
    pca = PCA(n_components=1)
    pca.fit(diffs)
    d = pca.components_[0]
    d = d / (np.linalg.norm(d) + 1e-8)
    return ProbeResult(direction=d)


PROBE_ALGORITHMS = {
    "DIM": train_dim,
    "LDA": train_lda,
    "LR": train_lr,
    "PCA": train_pca,
    "LAT": train_lat,
}

print(f"Probe algorithms: {list(PROBE_ALGORITHMS.keys())}")

In [ ]:
#######################################################################
# Evaluation avec AUC + 4-fold CV pour best layer
#######################################################################

def compute_auc(probe, X, y):
    """Compute ROC AUC, trying both directions (like George's code)."""
    logits = probe.predict(X)
    if len(np.unique(y)) < 2:
        return 0.5
    auc_normal = roc_auc_score(y, logits)
    auc_flipped = roc_auc_score(y, -logits)
    return max(auc_normal, auc_flipped)


def compute_accuracy(probe, X, y):
    """Compute accuracy, trying both directions."""
    logits = probe.predict(X)
    preds = logits > 0
    acc = accuracy_score(y, preds)
    acc_flipped = accuracy_score(y, ~preds)
    return max(acc, acc_flipped)


def select_best_layer_cv(all_acts, layer_names, algo_name, algo_fn, n_splits=4):
    """
    4-fold Stratified CV pour selectionner le best layer.
    Pas de data leakage : on ne touche PAS aux donnees de validation.
    Exactement comme George l'a implemente.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    best_layer = layer_names[0]
    best_auc = -1.0

    for layer_name in layer_names:
        valid = [s for s in all_acts if layer_name in s["activations"]]
        if len(valid) < n_splits * 2:
            continue

        X = np.stack([s["activations"][layer_name] for s in valid]).astype(np.float32)
        y = np.array([s["label"] for s in valid], dtype=bool)

        fold_aucs = []
        for train_idx, val_idx in cv.split(X, y):
            X_tr, X_va = X[train_idx], X[val_idx]
            y_tr, y_va = y[train_idx], y[val_idx]

            try:
                probe = algo_fn(X_tr, y_tr)
                logits = probe.predict(X_va)
                if len(np.unique(y_va)) < 2:
                    continue
                auc = roc_auc_score(y_va, logits)
                auc_flip = roc_auc_score(y_va, -logits)
                fold_aucs.append(max(auc, auc_flip))
            except Exception:
                continue

        mean_auc = np.mean(fold_aucs) if fold_aucs else 0.0
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_layer = layer_name

    return best_layer, best_auc


print("Evaluation functions ready (AUC + 4-fold CV)")

## 3. Lancer l'experience sur Llama-3.1-70B-Instruct

In [ ]:
#######################################################################
# MAIN EXPERIMENT
#######################################################################

model_name = MODEL_CONFIG["name"]
num_params = MODEL_CONFIG["num_params"]

print(f"{'='*60}")
print(f"  MODEL: {model_name} ({num_params}B params)")
print(f"  Metrique: AUC (ROC AUC)")
print(f"  Best layer: 4-fold Stratified CV")
print(f"  Started: {time.strftime('%H:%M:%S')}")
print(f"{'='*60}")

model_dir = Path(RESULTS_DIR) / model_name
model_dir.mkdir(parents=True, exist_ok=True)

# ---- Load model ----
start_time = time.time()
model, tokenizer, layers = load_model(MODEL_CONFIG)
num_layers = len(layers)

# ---- Extract activations for all datasets ----
all_activations = {}
for ds_config in DATASETS_CONFIG:
    ds_name = ds_config["name"]
    print(f"\n--- Dataset: {ds_name} ---")
    dataset = load_binary_dataset(ds_config)
    print(f"  Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}")

    ds_activations = {"train": [], "validation": []}
    for split in ["train", "validation"]:
        print(f"  Extracting {split}...")
        errors = 0
        for sample in tqdm(dataset[split], desc=f"    {split}"):
            try:
                acts = extract_activations(model, tokenizer, layers, sample["text"], LAYERS_SKIP)
                ds_activations[split].append({
                    "activations": acts,
                    "label": sample["label"],
                    "group_id": sample.get("group_id"),
                })
            except Exception as e:
                errors += 1
                if errors <= 2:
                    print(f"    Warning: {e}")
                continue
        if errors > 0:
            print(f"    ({errors} erreurs ignorees)")

    all_activations[ds_name] = ds_activations
    print(f"  OK: {len(ds_activations['train'])} train, {len(ds_activations['validation'])} val")

# ---- Unload model to free VRAM ----
del model, tokenizer, layers
gc.collect()
torch.cuda.empty_cache()
vram_after = torch.cuda.memory_allocated() / 1024**3
print(f"\nModel unloaded. VRAM: {vram_after:.1f} GB")

# ---- Train probes with 4-fold CV for best layer ----
print(f"\n{'='*60}")
print(f"  TRAINING PROBES (4-fold CV for best layer)")
print(f"{'='*60}")

all_results = []
cv_results = []  # Track 4-fold CV results

first_ds = list(all_activations.keys())[0]
first_samples = all_activations[first_ds]["train"]
layer_names = sorted(first_samples[0]["activations"].keys(), key=lambda x: int(x[1:]))
print(f"Layers disponibles: {layer_names}")

for train_ds in all_activations:
    train_samples = all_activations[train_ds]["train"]
    print(f"\n--- Train dataset: {train_ds} ---")

    for algo_name, algo_fn in PROBE_ALGORITHMS.items():
        # STEP 1: 4-fold CV sur le TRAIN set pour trouver le best layer
        best_layer, cv_auc = select_best_layer_cv(
            train_samples, layer_names, algo_name, algo_fn, n_splits=N_FOLDS
        )
        best_layer_idx = int(best_layer[1:])
        print(f"  {algo_name}: best layer = {best_layer_idx} (CV AUC = {cv_auc:.3f})")

        cv_results.append({
            "train_dataset": train_ds,
            "algorithm": algo_name,
            "best_layer": best_layer_idx,
            "cv_auc": cv_auc,
        })

        # STEP 2: Train sur TOUT le train set au best layer
        valid_train = [s for s in train_samples if best_layer in s["activations"]]
        if len(valid_train) < 10:
            continue
        X_train = np.stack([s["activations"][best_layer] for s in valid_train]).astype(np.float32)
        y_train = np.array([s["label"] for s in valid_train], dtype=bool)

        try:
            probe = algo_fn(X_train, y_train)
        except Exception:
            continue

        # STEP 3: Evaluer sur les VALIDATION sets (in-dist + cross-dataset)
        for eval_ds in all_activations:
            val_samples = all_activations[eval_ds]["validation"]
            valid_val = [s for s in val_samples if best_layer in s["activations"]]
            if len(valid_val) < 10:
                continue
            X_val = np.stack([s["activations"][best_layer] for s in valid_val]).astype(np.float32)
            y_val = np.array([s["label"] for s in valid_val], dtype=bool)

            try:
                auc = compute_auc(probe, X_val, y_val)
                acc = compute_accuracy(probe, X_val, y_val)
            except Exception:
                continue

            all_results.append({
                "model": model_name,
                "num_params": num_params,
                "train_dataset": train_ds,
                "eval_dataset": eval_ds,
                "algorithm": algo_name,
                "best_layer": best_layer_idx,
                "num_layers": num_layers,
                "auc": auc,
                "accuracy": acc,
                "cv_auc": cv_auc,
                "in_distribution": train_ds == eval_ds,
            })

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"  DONE in {elapsed/60:.1f} min")
print(f"  {len(all_results)} results")
print(f"{'='*60}")

# Save results
df = pd.DataFrame(all_results)
df.to_csv(f"{RESULTS_DIR}/results_llama70b.csv", index=False)

df_cv = pd.DataFrame(cv_results)
df_cv.to_csv(f"{RESULTS_DIR}/cv_results_llama70b.csv", index=False)

print(f"\nSaved to {RESULTS_DIR}/results_llama70b.csv")

## 4. Resultats: AUC par algorithme et dataset

In [ ]:
# Charger les resultats
df = pd.read_csv(f"{RESULTS_DIR}/results_llama70b.csv")
df_cv = pd.read_csv(f"{RESULTS_DIR}/cv_results_llama70b.csv")

print(f"Total: {len(df)} results")
print(f"Modele: Llama-2-70b-chat (70B params)")

# ---- Tableau recapitulatif AUC ----
in_dist = df[df['in_distribution'] == True]
print(f"\n{'='*60}")
print(f"  RESULTATS AUC - Llama-2-70b-chat")
print(f"{'='*60}")

# AUC par algorithme (in-distribution)
print("\n--- AUC par algorithme (in-distribution, moyenne sur datasets) ---")
algo_auc = in_dist.groupby('algorithm')['auc'].mean().sort_values(ascending=False)
for algo, auc in algo_auc.items():
    print(f"  {algo:6s}: AUC = {auc:.3f}")

# AUC par dataset (in-distribution, meilleur algo)
print("\n--- AUC par dataset (in-distribution, meilleur algo) ---")
for ds in in_dist['train_dataset'].unique():
    ds_data = in_dist[in_dist['train_dataset'] == ds]
    best_row = ds_data.loc[ds_data['auc'].idxmax()]
    print(f"  {ds:20s}: AUC = {best_row['auc']:.3f} (algo={best_row['algorithm']}, layer={int(best_row['best_layer'])})")

# AUC globale
print(f"\n  AUC GLOBALE (in-dist): {in_dist['auc'].mean():.3f}")
print(f"  Meilleur algo global: {algo_auc.index[0]} (AUC={algo_auc.iloc[0]:.3f})")

# 4-fold CV summary
print(f"\n--- 4-fold CV: best layer par algo/dataset ---")
print(df_cv.to_string(index=False))

In [ ]:
# Graphe 1: AUC par algorithme (in-distribution)
in_dist = df[df['in_distribution'] == True]
datasets = sorted(in_dist['train_dataset'].unique())
algorithms = sorted(in_dist['algorithm'].unique())
n_ds = len(datasets)
x = np.arange(len(algorithms))
width = 0.8 / max(n_ds, 1)
colors = plt.cm.Set2(np.linspace(0, 1, n_ds))

fig, ax = plt.subplots(figsize=(12, 7))
for i, ds in enumerate(datasets):
    ds_data = in_dist[in_dist['train_dataset'] == ds]
    algo_auc = ds_data.groupby('algorithm')['auc'].mean()
    vals = [algo_auc.get(a, 0.5) for a in algorithms]
    ax.bar(x + i * width, vals, width, label=ds, color=colors[i])

ax.set_xlabel('Probe Algorithm', fontsize=12)
ax.set_ylabel('AUC (ROC)', fontsize=12)
ax.set_title('Llama-2-70b-chat: AUC par Algorithme et Dataset (In-Distribution)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (n_ds - 1) / 2)
ax.set_xticklabels(algorithms)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0.3, 1.0)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random (0.5)')
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/llama70b_auc_by_algorithm.png", dpi=150)
plt.show()

In [ ]:
# Graphe 2: Matrice de generalisation AUC (cross-dataset)
best_algo = in_dist.groupby('algorithm')['auc'].mean().idxmax()
best_algo_data = df[df['algorithm'] == best_algo]

matrix = best_algo_data.groupby(['train_dataset', 'eval_dataset'])['auc'].mean().reset_index()
pivot = matrix.pivot(index='train_dataset', columns='eval_dataset', values='auc')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot.values, cmap='YlOrRd', vmin=0.4, vmax=1.0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(pivot.index, fontsize=9)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10,
                    color='white' if val > 0.75 else 'black')
ax.set_xlabel('Eval Dataset', fontsize=11)
ax.set_ylabel('Train Dataset', fontsize=11)
ax.set_title(f'Llama-2-70b-chat: Cross-Dataset AUC ({best_algo})', fontweight='bold', fontsize=13)
fig.colorbar(im, label='AUC')
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/llama70b_generalization_matrix.png", dpi=150)
plt.show()

In [ ]:
# Graphe 3: 4-fold CV AUC par layer (pour visualiser le choix du best layer)
# On refait le CV pour TOUTES les layers pour le meilleur algo, pour le graphe
print(f"Generating layer-wise CV AUC plot for best algo ({best_algo})...")

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
algo_fn = PROBE_ALGORITHMS[best_algo]

fig, axes = plt.subplots(1, len(DATASETS_CONFIG), figsize=(5 * len(DATASETS_CONFIG), 5), sharey=True)
if len(DATASETS_CONFIG) == 1:
    axes = [axes]

for ax, ds_name in zip(axes, [d['name'] for d in DATASETS_CONFIG]):
    if ds_name not in all_activations:
        continue
    train_samples = all_activations[ds_name]["train"]

    layer_aucs = {}
    for layer_name in layer_names:
        valid = [s for s in train_samples if layer_name in s["activations"]]
        if len(valid) < N_FOLDS * 2:
            continue
        X = np.stack([s["activations"][layer_name] for s in valid]).astype(np.float32)
        y = np.array([s["label"] for s in valid], dtype=bool)

        fold_aucs = []
        for train_idx, val_idx in cv.split(X, y):
            try:
                probe = algo_fn(X[train_idx], y[train_idx])
                logits = probe.predict(X[val_idx])
                auc = roc_auc_score(y[val_idx], logits)
                auc_flip = roc_auc_score(y[val_idx], -logits)
                fold_aucs.append(max(auc, auc_flip))
            except Exception:
                continue
        if fold_aucs:
            layer_aucs[int(layer_name[1:])] = np.mean(fold_aucs)

    if layer_aucs:
        layers_sorted = sorted(layer_aucs.keys())
        aucs_sorted = [layer_aucs[l] for l in layers_sorted]
        best_l = max(layer_aucs, key=layer_aucs.get)
        ax.plot(layers_sorted, aucs_sorted, 'b-o', markersize=4, linewidth=2)
        ax.axvline(x=best_l, color='red', linestyle='--', alpha=0.7, label=f'Best: layer {best_l}')
        ax.legend(fontsize=9)

    ax.set_title(ds_name, fontweight='bold')
    ax.set_xlabel('Layer')
    ax.set_ylim(0.4, 1.0)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel(f'4-fold CV AUC ({best_algo})')
fig.suptitle('Llama-2-70b-chat: Layer Selection via 4-fold CV', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/llama70b_layer_cv_auc.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tableau final: tout ensemble
print(f"\n{'='*60}")
print(f"  RESUME FINAL - Llama-2-70b-chat (70B)")
print(f"{'='*60}")

# In-distribution AUC
print("\n=== IN-DISTRIBUTION ===")
for ds in sorted(in_dist['train_dataset'].unique()):
    ds_data = in_dist[in_dist['train_dataset'] == ds]
    best = ds_data.loc[ds_data['auc'].idxmax()]
    print(f"  {ds:20s}  AUC={best['auc']:.3f}  Acc={best['accuracy']:.1%}  Algo={best['algorithm']}  Layer={int(best['best_layer'])}")

print(f"\n  MOYENNE IN-DIST:     AUC={in_dist['auc'].mean():.3f}  Acc={in_dist['accuracy'].mean():.1%}")

# Cross-dataset AUC
cross = df[df['in_distribution'] == False]
if len(cross) > 0:
    print(f"\n=== CROSS-DATASET (generalisation) ===")
    print(f"  MOYENNE CROSS-DS:    AUC={cross['auc'].mean():.3f}  Acc={cross['accuracy'].mean():.1%}")

# Save summary
summary = {
    "model": model_name,
    "params": "70B",
    "in_dist_auc": float(in_dist['auc'].mean()),
    "in_dist_acc": float(in_dist['accuracy'].mean()),
    "cross_ds_auc": float(cross['auc'].mean()) if len(cross) > 0 else None,
    "cross_ds_acc": float(cross['accuracy'].mean()) if len(cross) > 0 else None,
    "best_algo": in_dist.groupby('algorithm')['auc'].mean().idxmax(),
    "n_folds": N_FOLDS,
}
with open(f"{RESULTS_DIR}/summary_llama70b.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved to {RESULTS_DIR}/summary_llama70b.json")

## 5. Fichiers sauvegardes

Tous les resultats sont dans ton Google Drive:
- `LLM_Lie_Detection_Results/results_llama70b.csv` — tous les resultats detailles
- `LLM_Lie_Detection_Results/cv_results_llama70b.csv` — resultats du 4-fold CV
- `LLM_Lie_Detection_Results/summary_llama70b.json` — resume
- `LLM_Lie_Detection_Results/*.png` — graphiques

In [ ]:
print("Fichiers sauvegardes dans Google Drive:")
for f in sorted(Path(RESULTS_DIR).glob('*llama*')):
    size = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size:8.1f} KB")
print(f"\nChemin: {RESULTS_DIR}")
print("\nFini! Tu peux telecharger les fichiers depuis Google Drive.")